# ⚠️ LEAKAGE-SAFE corrected notebook — code-reviewed, NOT yet executed

This is the leakage-safe version. All data-dependent preprocessing (95th-percentile
threshold, StandardScaler, PCA, angle normalization) is now fit on **training data only**,
and under cross-validation it is **refit inside every fold** (and, for LOCO-CV, on the
training clusters only). Seeds, sample sizes, split ratios, models, and hyperparameters
are unchanged.

The changed cells are marked `LEAKAGE-SAFE`. **All outputs were cleared** — the previous
cached outputs came from the leaky pipeline and are not valid here. Run this on Kaggle
(with the datasets attached) to generate the corrected results, then save the CSVs and
prediction files under a new name. Do not treat any number as corrected until this notebook
has actually been executed.


# QDFL Hybrid v2 — Cross-Validated Evaluation

This notebook reuses the **exact data pipeline, preprocessing, and model definitions** of `qdfl-hybrid-v2-kaggle-singleseed1.ipynb` and replaces the single-seed train/test evaluation with **rigorous cross-validation**, applied per model family:

| Family | Cross-validation scheme |
|---|---|
| Classical ML (LogReg, RF, XGBoost, NN) | **Stratified K-Fold** + **Repeated Stratified K-Fold** |
| Quantum ML (VQC, QDCN, QSVM) | **Repeated K-Fold** + **Monte-Carlo Holdout** |
| Federated Learning (FedAvg) | **Group K-Fold** + **LOCO-CV** (leave-one-cluster-out) |
| QDFL (proposed) | **LOCO-CV + Stratified-Federated-CV combo** |

All metrics are reported as **mean ± std across folds** (ROC-AUC, F1, Average-Precision).

> **Runtime note.** Quantum and quantum-federated CV are simulator-heavy. The fold counts and sub-sample sizes below are set to *tractable* defaults for a Kaggle GPU session and are exposed as constants at the top of each section so they can be scaled up. Each section is self-contained and can be run independently after the setup/definition cells.

## Part A — Setup, data, features, preprocessing (reused unchanged)

In [1]:
# ─── Install PennyLane (Kaggle pre-installs torch, xgboost, sklearn, pandas) ─
# We do NOT touch numpy. Kaggle ships numpy 2.x and every other library that
# came pre-installed was compiled against it. Downgrading numpy breaks the
# whole stack with "dtype size changed" ABI errors, which is what happened
# in the previous version of this cell.
#
# --upgrade-strategy only-if-needed prevents pip from greedily upgrading
# transitive deps. If pip somehow still upgrades numpy/scipy and the next
# cell fails with an ABI error, use Kaggle's "Factory reset" (Run → Session
# options → Factory reset) and rerun from a clean session.

!pip install -q --upgrade-strategy only-if-needed pennylane pennylane-lightning gdown
import warnings; warnings.filterwarnings("ignore")
print("✓ Installation complete — proceeding to imports")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 8.0 MB/s eta 0:00:00
✓ Installation complete — proceeding to imports


In [2]:
import os, time, copy, random, json
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display   # explicit, so display() never NameErrors

# ── Classical ML ─────────────────────────────────────────────────────────
from sklearn.linear_model      import LogisticRegression
from sklearn.ensemble          import RandomForestClassifier
from sklearn.svm               import SVC
from sklearn.preprocessing     import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.decomposition     import PCA
from sklearn.model_selection   import train_test_split
from sklearn.metrics           import (roc_auc_score, f1_score,
                                       average_precision_score)
from scipy.stats               import spearmanr
import xgboost as xgb

# ── PyTorch / Quantum / FL ───────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pennylane as qml

# ── Single-seed configuration ────────────────────────────────────────────
# This notebook runs with ONE fixed seed for reproducibility and to fit
# Kaggle's GPU-time quota. (The multi-seed mean ± std version is heavier:
# 3 seeds x 2 datasets x 2 splits of QDFL is the 4-6 h bottleneck. Single
# seed cuts that to roughly a third.)
SEED = 42

def set_global_seed(seed: int):
    np.random.seed(seed); torch.manual_seed(seed); random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_global_seed(SEED)
# Note: `device` is set in the CUDA self-test cell that runs next.
print(f"Seed          : {SEED}")
print(f"PennyLane     : {qml.__version__}")
print(f"PyTorch       : {torch.__version__}")
print(f"XGBoost       : {xgb.__version__}")

Seed          : 42
PennyLane     : 0.45.1
PyTorch       : 2.10.0+cu128
XGBoost       : 3.2.0


In [3]:
# ─── CUDA self-test (diagnostic — catches "no kernel image" errors early) ──
import subprocess
print("=" * 70)
print("CUDA DIAGNOSTIC")
print("=" * 70)

# Show what GPU Kaggle gave us
try:
    smi = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total",
                          "--format=csv,noheader"], capture_output=True, text=True, timeout=5)
    print(f"GPU         : {smi.stdout.strip() or '(not detected)'}")
except Exception as e:
    print(f"nvidia-smi  : {e}")

print(f"PyTorch     : {torch.__version__}")
print(f"PyTorch CUDA: {torch.version.cuda}")
print(f"cuDNN       : {torch.backends.cudnn.version()}")
print(f"is_available: {torch.cuda.is_available()}")

# torch.cuda.is_available() returns True even when the kernel image is missing
# for the device, so we need a real op to confirm dispatch works.
cuda_works = False
if torch.cuda.is_available():
    try:
        cap = torch.cuda.get_device_capability(0)
        name = torch.cuda.get_device_name(0)
        print(f"GPU name    : {name}")
        print(f"Compute cap : sm_{cap[0]}{cap[1]}")
        # Real kernel launch
        x = torch.randn(64, 64, device="cuda")
        y = x @ x.T
        _ = y.sum().item()        # synchronise
        cuda_works = True
        print("✓ Kernel launch succeeded — CUDA is functional")
    except RuntimeError as e:
        print(f"✗ Kernel launch FAILED: {str(e)[:200]}")

# Always set device — fall back to CPU if CUDA is not functional. This avoids
# blocking the rest of the notebook (the user can still develop on CPU even if
# GPU is unavailable; quantum cells will just run more slowly).
if cuda_works:
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
    print()
    print("⚠ CUDA is NOT functional — falling back to CPU.")
    print("  Quantum-model training will be slow but will still run.")
    print("  If you need GPU acceleration:")
    print("  1. Check the right sidebar — Accelerator must be GPU T4 x2 or P100.")
    print("  2. If it was already on GPU, do a Factory reset (Run → Session")
    print("     options → Factory reset) and re-run from cell 1.")
    print("  3. As a last resort, uncomment and run the recovery cell below.")

print(f"\n✓ Selected device: {device}")

CUDA DIAGNOSTIC
GPU         : Tesla P100-PCIE-16GB, 580.159.04, 16384 MiB
PyTorch     : 2.10.0+cu128
PyTorch CUDA: 12.8
cuDNN       : 91002
is_available: True
GPU name    : Tesla P100-PCIE-16GB
Compute cap : sm_60
✗ Kernel launch FAILED: CUDA error: no kernel image is available for execution on the device
Search for `cudaErrorNoKernelImageForDevice' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more in

⚠ CUDA is NOT functional — falling back to CPU.
  Quantum-model training will be slow but will still run.
  If you need GPU acceleration:
  1. Check the right sidebar — Accelerator must be GPU T4 x2 or P100.
  2. If it was already on GPU, do a Factory reset (Run → Session
     options → Factory reset) and re-run from cell 1.
  3. As a last resort, uncomment and run the recovery cell below.

✓ Selected device: cpu


In [4]:
import gdown
import glob

# ─── Kaggle-aware data resolution ──────────────────────────────────────────
# Working dir on Kaggle is /kaggle/working (writable, 20 GB).
# Attached datasets live under /kaggle/input/<dataset-slug>/ (read-only).
IS_KAGGLE = os.path.exists('/kaggle')
DATA_DIR  = '/kaggle/working/data' if IS_KAGGLE else '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)

PRIMARY_PATH    = f'{DATA_DIR}/primary.csv'
COMPARISON_PATH = f'{DATA_DIR}/comparison.csv'

# ─── PaySim: attempt to load from attached Kaggle dataset first ────────────
# If you attached `ealaxi/paysim1` in the right sidebar (Add Data),
# it will appear at /kaggle/input/paysim1/PS_20174392719_1491204439457_log.csv
def find_paysim_in_kaggle_input():
    if not IS_KAGGLE: return None
    candidates = glob.glob('/kaggle/input/**/PS_*.csv', recursive=True)
    if not candidates:
        # Sometimes the filename is different; look for any .csv in a paysim dir
        candidates = glob.glob('/kaggle/input/*paysim*/*.csv') + \
                     glob.glob('/kaggle/input/*PaySim*/*.csv')
    return candidates[0] if candidates else None

def find_azamuke_in_kaggle_input():
    if not IS_KAGGLE: return None
    # Azamuke 2024 mobile money dataset — user may have uploaded it
    # Look for any CSV with 'azamuke' or 'mobile' in path
    for pattern in ['/kaggle/input/*azamuke*/*.csv',
                    '/kaggle/input/*mobile*money*/*.csv',
                    '/kaggle/input/*synthetic*mobile*/*.csv']:
        m = glob.glob(pattern)
        if m: return m[0]
    return None

# ─── Resolve PaySim ────────────────────────────────────────────────────────
paysim_attached = find_paysim_in_kaggle_input()
if paysim_attached:
    if not os.path.exists(COMPARISON_PATH):
        import shutil; shutil.copy(paysim_attached, COMPARISON_PATH)
    print(f'✓ PaySim loaded from attached Kaggle dataset: {paysim_attached}')
else:
    # Fallback: download via gdown (requires Internet On)
    COMPARISON_ID = '1RyHztdRMYei4N3PBHLGnwgL0Lrfrfmx9'  # PaySim
    if not os.path.exists(COMPARISON_PATH):
        print('PaySim not attached — downloading from Google Drive (requires Internet)')
        gdown.download(id=COMPARISON_ID, output=COMPARISON_PATH, quiet=False, fuzzy=True)

# ─── Resolve Azamuke 2024 ──────────────────────────────────────────────────
azamuke_attached = find_azamuke_in_kaggle_input()
if azamuke_attached:
    if not os.path.exists(PRIMARY_PATH):
        import shutil; shutil.copy(azamuke_attached, PRIMARY_PATH)
    print(f'✓ Azamuke 2024 loaded from attached Kaggle dataset: {azamuke_attached}')
else:
    PRIMARY_ID = '12p6v5_1fFCKUJgO2gnrKMr013pUNQmtf'  # Azamuke 2024
    if not os.path.exists(PRIMARY_PATH):
        print('Azamuke not attached — downloading from Google Drive (requires Internet)')
        gdown.download(id=PRIMARY_ID, output=PRIMARY_PATH, quiet=False, fuzzy=True)

print()
print(f'Primary    : {os.path.getsize(PRIMARY_PATH)/1e6:7.1f} MB at {PRIMARY_PATH}')
print(f'Comparison : {os.path.getsize(COMPARISON_PATH)/1e6:7.1f} MB at {COMPARISON_PATH}')

PaySim not attached — downloading from Google Drive (requires Internet)


Downloading...
From (original): https://drive.google.com/uc?id=1RyHztdRMYei4N3PBHLGnwgL0Lrfrfmx9
From (redirected): https://drive.google.com/uc?id=1RyHztdRMYei4N3PBHLGnwgL0Lrfrfmx9&confirm=t&uuid=6a8201ed-00ee-4f17-a456-e5df6558f8cf
To: /kaggle/working/data/comparison.csv
100%|██████████| 494M/494M [00:03<00:00, 131MB/s]


Azamuke not attached — downloading from Google Drive (requires Internet)


Downloading...
From (original): https://drive.google.com/uc?id=12p6v5_1fFCKUJgO2gnrKMr013pUNQmtf
From (redirected): https://drive.google.com/uc?id=12p6v5_1fFCKUJgO2gnrKMr013pUNQmtf&confirm=t&uuid=eecd8c0a-8acc-4d5a-9c62-186c366ddcca
To: /kaggle/working/data/primary.csv
100%|██████████| 157M/157M [00:01<00:00, 111MB/s]


Primary    :   156.6 MB at /kaggle/working/data/primary.csv
Comparison :   493.5 MB at /kaggle/working/data/comparison.csv


In [5]:
STD_COLS = ['txType', 'amount', 'oldBalSender', 'newBalSender',
            'oldBalRecipient', 'newBalRecipient', 'step', 'isFraud']

AZAMUKE_RENAME = {
    'transactionType' : 'txType',
    'oldBalInitiator' : 'oldBalSender',
    'newBalInitiator' : 'newBalSender',
}
PAYSIM_RENAME = {
    'type'           : 'txType',
    'oldbalanceOrg'  : 'oldBalSender',
    'newbalanceOrig' : 'newBalSender',
    'oldbalanceDest' : 'oldBalRecipient',
    'newbalanceDest' : 'newBalRecipient',
}

def load_dataset(path, name):
    df = pd.read_csv(path)
    cols = set(df.columns)
    if 'transactionType' in cols:
        schema = 'Azamuke 2024'
        df = df.rename(columns=AZAMUKE_RENAME)
    elif 'type' in cols and 'oldbalanceOrg' in cols:
        schema = 'PaySim (Lopez-Rojas)'
        df = df.rename(columns=PAYSIM_RENAME)
        df = df.drop(columns=[c for c in ('nameOrig','nameDest','isFlaggedFraud')
                              if c in df.columns])
    else:
        raise ValueError(
            f"Unrecognised schema for '{name}' at {path}. "
            f"Columns found: {sorted(cols)}. Expected either Azamuke "
            f"('transactionType', ...) or PaySim ('type', 'oldbalanceOrg', ...). "
            f"Update AZAMUKE_RENAME / PAYSIM_RENAME or the detection logic.")
    missing = [c for c in STD_COLS if c not in df.columns]
    if missing:
        raise KeyError(
            f"'{name}' ({schema}) is missing required columns after renaming: "
            f"{missing}. Present columns: {sorted(df.columns)}. "
            f"Adjust the rename map so these standard names are produced.")
    df = df[STD_COLS].copy()
    print(f'{name:30s} | schema={schema:25s} | '
          f'{len(df):>10,} rows | fraud {df["isFraud"].mean()*100:5.2f}%')
    return df

df_primary = load_dataset(PRIMARY_PATH,    'PRIMARY')
df_paysim  = load_dataset(COMPARISON_PATH, 'COMPARISON')

DATASETS = {
    'Primary (Azamuke 2024)' : df_primary,
    'PaySim (Lopez-Rojas)'   : df_paysim,
}

PRIMARY                        | schema=Azamuke 2024              |  1,720,181 rows | fraud 10.20%
COMPARISON                     | schema=PaySim (Lopez-Rojas)      |  6,362,620 rows | fraud  0.13%


In [6]:
# ============================================================================
# LEAKAGE-SAFE PATCH — reusable preprocessor (fit on TRAIN rows only).
# Generalises the deployment notebooks' AnglePP/StdPP. See README.
# ============================================================================
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.decomposition import PCA

ROWWISE = ['txType_enc','logAmount','oldBalSender','newBalSender','oldBalRecipient',
           'newBalRecipient','balDiffSender','balDiffRecipient','amtToOldBalRatio',
           'zeroOldBalSender','zeroNewBalSender','senderDrained','hourOfDay','dayOfWeek','step']
FEATURE_ORDER = ROWWISE + ['largeTransaction']   # 16 features

class LeakageSafePreprocessor:
    """Fit threshold, scaler, PCA and angle-normaliser on TRAINING rows only."""
    def __init__(self, mode='angle', n_components=8, seed=42):
        assert mode in ('angle','std'); self.mode=mode; self.nc=n_components; self.seed=seed
    def _matrix(self, df):
        out = df[ROWWISE].copy()
        out['largeTransaction'] = (df['amount'].values > self.thr).astype(int)
        return out[FEATURE_ORDER].values.astype(np.float32)
    def fit(self, tr):
        self.thr = float(tr['amount'].quantile(0.95))          # threshold <- TRAIN
        X = self._matrix(tr); self.sc = StandardScaler().fit(X); X = self.sc.transform(X)
        if self.mode == 'angle':
            self.pca = PCA(self.nc, random_state=self.seed).fit(X); X = self.pca.transform(X)
            self.mm = MinMaxScaler((-np.pi, np.pi)).fit(X)
        return self
    def transform(self, df):
        X = self.sc.transform(self._matrix(df))
        if self.mode == 'angle': X = self.mm.transform(self.pca.transform(X))
        return X.astype(np.float32)


# ---- Deterministic ROW-LEVEL feature engineering (leakage-safe to run pre-split) ----
# CHANGED (leakage-safe): `largeTransaction` is NO LONGER computed here from a global
# 95th-percentile. Its threshold is learned from the TRAINING split inside
# LeakageSafePreprocessor. Raw `amount` is kept so the preprocessor can compute it.
TARGET = 'isFraud'
def engineer_features(df_in):
    df = df_in.copy()
    df['txType_enc']       = LabelEncoder().fit_transform(df['txType'])
    df['logAmount']        = np.log1p(df['amount'])
    df['balDiffSender']    = df['newBalSender']    - df['oldBalSender']
    df['balDiffRecipient'] = df['newBalRecipient'] - df['oldBalRecipient']
    df['amtToOldBalRatio'] = df['amount'] / (df['oldBalSender'] + 1e-6)
    df['zeroOldBalSender'] = (df['oldBalSender']==0).astype(int)
    df['zeroNewBalSender'] = (df['newBalSender']==0).astype(int)
    df['senderDrained']    = ((df['newBalSender']==0)&(df['oldBalSender']>0)).astype(int)
    df['hourOfDay']        = df['step'] % 24
    df['dayOfWeek']        = (df['step']//24) % 7
    return df[ROWWISE + ['amount', TARGET]].copy()   # raw rows + amount + label
DATASETS_ENG = {n: engineer_features(d) for n,d in DATASETS.items()}
for n,d in DATASETS_ENG.items(): print(f'{n:25s} | {len(d):>10,} rows | row-level features ready')


Primary (Azamuke 2024)    |  1,720,181 rows | row-level features ready
PaySim (Lopez-Rojas)      |  6,362,620 rows | row-level features ready


In [7]:
# ---- Data preparation: SPLIT FIRST, then fit preprocessing on TRAIN only ----
# CHANGED (leakage-safe): the OLD prepare_dataset fit StandardScaler/PCA/MinMax on the
# FULL balanced pool before train_test_split. This version subsamples, SPLITS, then fits
# LeakageSafePreprocessor on the TRAIN split and transforms held-out data. Downstream
# cells are unchanged: the returned dict keys are identical.
SAMPLE_QUANT, SAMPLE_QSVM, SAMPLE_CLASS, N_QUBITS = 5000, 700, 16000, 8

def _subsample(df_eng, n, seed):
    fr = df_eng[df_eng[TARGET]==1].sample(n=n//2, random_state=seed).index
    lg = df_eng[df_eng[TARGET]==0].sample(n=n//2, random_state=seed).index
    return df_eng.loc[fr.tolist()+lg.tolist()].sample(frac=1, random_state=seed)

def prepare_dataset(df_eng, name, seed):
    set_global_seed(seed); out = {'name': name, 'seed': seed}
    def split_fit(df_sub, mode, ncomp, test_size, val_size=None):
        tr, te = train_test_split(df_sub, test_size=test_size, stratify=df_sub[TARGET], random_state=seed)
        va = None
        if val_size is not None:
            tr, va = train_test_split(tr, test_size=val_size, stratify=tr[TARGET], random_state=seed)
        pp = LeakageSafePreprocessor(mode, ncomp, seed).fit(tr)        # TRAIN ONLY
        d = {'tr': (pp.transform(tr), tr[TARGET].values.astype(int)),
             'te': (pp.transform(te), te[TARGET].values.astype(int))}
        if va is not None: d['va'] = (pp.transform(va), va[TARGET].values.astype(int))
        return d
    q = split_fit(_subsample(df_eng, SAMPLE_QUANT, seed),   'angle', N_QUBITS, 0.2, 0.15)
    out.update({'X_tr_q':q['tr'][0],'y_tr_q':q['tr'][1],'X_va_q':q['va'][0],'y_va_q':q['va'][1],
                'X_te_q':q['te'][0],'y_te_q':q['te'][1]})
    s = split_fit(_subsample(df_eng, SAMPLE_QSVM, seed+1),  'angle', 4, 0.286)
    out.update({'X_tr_qs':s['tr'][0],'y_tr_qs':s['tr'][1],'X_te_qs':s['te'][0],'y_te_qs':s['te'][1]})
    c = split_fit(_subsample(df_eng, SAMPLE_CLASS, seed+2), 'std', None, 0.2)
    out.update({'X_tr_c':c['tr'][0],'y_tr_c':c['tr'][1],'X_te_c':c['te'][0],'y_te_c':c['te'][1]})
    qc = split_fit(_subsample(df_eng, SAMPLE_CLASS, seed+2),'angle', N_QUBITS, 0.2)
    out.update({'X_tr_qc':qc['tr'][0],'y_tr_qc':qc['tr'][1],'X_te_qc':qc['te'][0],'y_te_qc':qc['te'][1]})
    return out

print(f'─── Preparing data (leakage-safe) for seed={SEED} ───')
DATA = {n: prepare_dataset(d, n, SEED) for n, d in DATASETS_ENG.items()}
print('✅ Data ready (train-only preprocessing)')


─── Preparing data (leakage-safe) for seed=42 ───
✅ Data ready (train-only preprocessing)


## Part B — Model definitions (reused; single-seed run-loops removed)

In [8]:
class SimpleClassicalNN(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.bn  = nn.BatchNorm1d(in_dim)
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(128,     64), nn.GELU(), nn.Dropout(0.2),
            nn.Linear( 64,      1), nn.Sigmoid())
    def forward(self, x):
        return self.net(self.bn(x)).squeeze(-1)

def train_nn_classical(X_tr, y_tr, X_te, y_te, name, seed,
                       epochs=30, batch=64, lr=1e-3):
    set_global_seed(seed)
    t0 = time.time()
    model = SimpleClassicalNN(X_tr.shape[1]).to(device)
    crit  = nn.BCELoss()
    opt   = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sch   = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    X_t, X_v, y_t, y_v = train_test_split(X_tr, y_tr, test_size=0.15,
                                          stratify=y_tr, random_state=seed)
    ld = DataLoader(TensorDataset(torch.tensor(X_t, dtype=torch.float32),
                                  torch.tensor(y_t, dtype=torch.float32)),
                    batch_size=batch, shuffle=True)
    best_auc, best_state = 0, None
    for _ in range(epochs):
        model.train()
        for Xb, yb in ld:
            Xb, yb = Xb.to(device), yb.to(device)
            opt.zero_grad(); crit(model(Xb), yb).backward(); opt.step()
        sch.step()
        model.eval()
        with torch.no_grad():
            v = model(torch.tensor(X_v, dtype=torch.float32).to(device)).cpu().numpy()
        v_auc = roc_auc_score(y_v, v)
        if v_auc > best_auc:
            best_auc = v_auc; best_state = copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        prob = model(torch.tensor(X_te, dtype=torch.float32).to(device)).cpu().numpy()
    pred = (prob >= 0.5).astype(int)
    return {
        'Dataset': name, 'Family': 'Classical', 'Model': 'Neural Network', 'Seed': seed,
        'ROC-AUC' : roc_auc_score(y_te, prob),
        'F1-Score': f1_score(y_te, pred),
        'Avg-Prec': average_precision_score(y_te, prob),
        'Params'  : sum(p.numel() for p in model.parameters()),
        'Time (s)': round(time.time()-t0, 1),
    }

def count_classical_params(clf):
    """Approximate trainable-parameter count for any sklearn-style classifier.
    Handles LR (coef + intercept), RF (sum of tree node counts), and
    XGBoost (sum of node counts across boosted trees).
    """
    # Linear models: coef_ + intercept_
    if hasattr(clf, "coef_"):
        n_coef = int(clf.coef_.size)
        n_intc = int(np.atleast_1d(getattr(clf, "intercept_", 0)).size)
        return n_coef + n_intc
    # sklearn ensembles: estimators_ holds DecisionTreeClassifier objects with tree_.node_count
    if hasattr(clf, "estimators_") and len(clf.estimators_) > 0 and hasattr(clf.estimators_[0], "tree_"):
        return int(sum(t.tree_.node_count for t in clf.estimators_))
    # XGBoost: count nodes across all boosted trees in the dumped booster
    if hasattr(clf, "get_booster"):
        try:
            trees = clf.get_booster().get_dump()
            return int(sum(t.count("\n") + 1 for t in trees))  # newline count ≈ node count
        except Exception:
            return int(getattr(clf, "n_estimators", 0))
    return 0


In [9]:
def make_loaders(Xtr, ytr, Xva, yva, batch=32):
    tr = TensorDataset(torch.tensor(Xtr, dtype=torch.float32),
                       torch.tensor(ytr, dtype=torch.float32))
    va = TensorDataset(torch.tensor(Xva, dtype=torch.float32),
                       torch.tensor(yva, dtype=torch.float32))
    _drop = (len(Xtr) % batch == 1)   # BatchNorm1d needs >1 sample per batch in train mode
    return (DataLoader(tr, batch_size=batch, shuffle=True, drop_last=_drop),
            DataLoader(va, batch_size=batch))

def train_quantum(model, tr_ld, va_ld, epochs=20, lr=5e-4, tag='Q', verbose=True):
    crit = nn.BCELoss()
    opt  = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sch  = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    best_auc, best_state = 0, None
    for ep in range(1, epochs+1):
        model.train()
        for Xb, yb in tr_ld:
            Xb, yb = Xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = crit(model(Xb), yb); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sch.step()
        model.eval(); ps, ls = [], []
        with torch.no_grad():
            for Xb, yb in va_ld:
                Xb = Xb.to(device); ls.extend(yb.numpy())
                ps.extend(model(Xb).cpu().numpy())
        va_auc = roc_auc_score(ls, ps)
        if va_auc > best_auc:
            best_auc = va_auc; best_state = copy.deepcopy(model.state_dict())
        if verbose and (ep % 5 == 0 or ep == 1):
            print(f'  [{tag}] Epoch {ep:>3}/{epochs} | AUC {va_auc:.4f}')
    model.load_state_dict(best_state)
    if verbose: print(f'  Best val AUC: {best_auc:.4f}')
    return best_auc

In [10]:
# ─── VQC (8-qubit, 3-layer StronglyEntanglingLayers, single PauliZ) ──────
N_LAYERS_VQC = 3

def build_vqc(seed):
    set_global_seed(seed)
    dev = qml.device('lightning.qubit', wires=N_QUBITS)
    @qml.qnode(dev, interface='torch', diff_method='best')
    def circuit(inputs, weights):
        qml.AngleEmbedding(inputs, wires=range(N_QUBITS), rotation='Y')
        qml.StronglyEntanglingLayers(weights, wires=range(N_QUBITS))
        return qml.expval(qml.PauliZ(0))
    qlayer = qml.qnn.TorchLayer(circuit, {'weights': (N_LAYERS_VQC, N_QUBITS, 3)})
    class VQC(nn.Module):
        def __init__(self):
            super().__init__()
            self.bn   = nn.BatchNorm1d(N_QUBITS)
            self.qnn  = qlayer
            self.post = nn.Sequential(
                nn.Linear(1, 16), nn.GELU(), nn.Dropout(0.2),
                nn.Linear(16, 1), nn.Sigmoid())
        def forward(self, x):
            x = self.bn(x); q = self.qnn(x).unsqueeze(-1)
            return self.post(q).squeeze(-1)
    return VQC().to(device)


In [11]:
# ─── QSVM (4-qubit angle-embedding fidelity kernel) ────────────────────
N_QUBITS_QSVM = 4

def build_qsvm_kernel(seed):
    set_global_seed(seed)
    dev_k = qml.device('lightning.qubit', wires=N_QUBITS_QSVM)
    @qml.qnode(dev_k)
    def kernel_circuit(x1, x2):
        qml.AngleEmbedding(x1, wires=range(N_QUBITS_QSVM), rotation='Y')
        qml.adjoint(qml.AngleEmbedding)(x2, wires=range(N_QUBITS_QSVM), rotation='Y')
        return qml.probs(wires=range(N_QUBITS_QSVM))
    def K(X1, X2):
        m, n = len(X1), len(X2)
        M = np.zeros((m, n))
        for i in range(m):
            for j in range(n):
                M[i,j] = float(kernel_circuit(X1[i], X2[j])[0])
        return M
    return K


In [12]:
# ─── QDCN standalone (proposed dual-channel; 2 layers per channel) ─────
N_Q_CH       = 4
N_ENT_LAYERS = 2

def build_qdcn(seed):
    set_global_seed(seed)
    dev_a = qml.device('lightning.qubit', wires=N_Q_CH)
    dev_b = qml.device('lightning.qubit', wires=N_Q_CH)
    @qml.qnode(dev_a, interface='torch', diff_method='best')
    def channel_a(inputs, weights):
        qml.AngleEmbedding(inputs, wires=range(N_Q_CH), rotation='Y')
        qml.BasicEntanglerLayers(weights, wires=range(N_Q_CH))
        return [qml.expval(qml.PauliZ(i)) for i in range(N_Q_CH)]
    @qml.qnode(dev_b, interface='torch', diff_method='best')
    def channel_b(inputs, weights):
        qml.AngleEmbedding(inputs, wires=range(N_Q_CH), rotation='X')
        for i in range(N_Q_CH-1):
            qml.CNOT(wires=[i, i+1])
        qml.BasicEntanglerLayers(weights, wires=range(N_Q_CH))
        return [qml.expval(qml.PauliY(i)) for i in range(N_Q_CH)]
    sa = {'weights': qml.BasicEntanglerLayers.shape(n_layers=N_ENT_LAYERS, n_wires=N_Q_CH)}
    sb = {'weights': qml.BasicEntanglerLayers.shape(n_layers=N_ENT_LAYERS, n_wires=N_Q_CH)}
    qa = qml.qnn.TorchLayer(channel_a, sa)
    qb = qml.qnn.TorchLayer(channel_b, sb)
    class QDCN(nn.Module):
        def __init__(self):
            super().__init__()
            self.bn     = nn.BatchNorm1d(N_QUBITS)
            self.q_a    = qa
            self.q_b    = qb
            self.post_a = nn.Sequential(nn.Linear(N_Q_CH, 16), nn.GELU(), nn.Dropout(0.2))
            self.post_b = nn.Sequential(nn.Linear(N_Q_CH, 16), nn.GELU(), nn.Dropout(0.2))
            self.fusion = nn.Sequential(
                nn.Linear(32, 64), nn.ReLU(), nn.Dropout(0.3),
                nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.2),
                nn.Linear(32,  1), nn.Sigmoid())
        def forward(self, x):
            x  = self.bn(x); xa = x[:, :N_Q_CH]; xb = x[:, N_Q_CH:]
            out = torch.cat([self.post_a(self.q_a(xa)),
                             self.post_b(self.q_b(xb))], dim=-1)
            return self.fusion(out).squeeze(-1)
    return QDCN().to(device)


In [13]:
class FraudDetectorNet(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.bn_in = nn.BatchNorm1d(in_dim)
        self.blk1  = nn.Sequential(nn.Linear(in_dim, 128), nn.BatchNorm1d(128), nn.GELU(), nn.Dropout(0.3))
        self.skip1 = nn.Linear(in_dim, 128)
        self.blk2  = nn.Sequential(nn.Linear(128, 64), nn.BatchNorm1d(64), nn.GELU(), nn.Dropout(0.25))
        self.skip2 = nn.Linear(128, 64)
        self.blk3  = nn.Sequential(nn.Linear(64, 32), nn.GELU(), nn.Dropout(0.2))
        self.head  = nn.Sequential(nn.Linear(32, 16), nn.ReLU(), nn.Linear(16, 1), nn.Sigmoid())
    def forward(self, x):
        x  = self.bn_in(x)
        x1 = self.blk1(x)  + self.skip1(x)
        x2 = self.blk2(x1) + self.skip2(x1)
        return self.head(self.blk3(x2)).squeeze(-1)

def weighted_fedavg_arrays(param_list, weights):
    w_sum = sum(weights)
    return [sum(w * p for w, p in zip(weights, layer)) / w_sum
            for layer in zip(*param_list)]

def split_clients(X, y, n_clients, mode='iid', alpha=0.5, seed=42):
    """IID stratified split or Dirichlet α-controlled label-skew split."""
    rng = np.random.default_rng(seed)
    if mode == 'iid':
        idx = np.arange(len(y))
        rng.shuffle(idx)
        return [(X[i], y[i]) for i in np.array_split(idx, n_clients)]
    # Dirichlet label-skew
    client_idx = [[] for _ in range(n_clients)]
    for cls in np.unique(y):
        cls_idx = np.where(y==cls)[0]
        rng.shuffle(cls_idx)
        prop = rng.dirichlet(np.repeat(alpha, n_clients))
        cuts = (np.cumsum(prop)[:-1] * len(cls_idx)).astype(int)
        for c, chunk in enumerate(np.split(cls_idx, cuts)):
            client_idx[c].extend(chunk.tolist())
    return [(X[np.array(i)], y[np.array(i)]) for i in client_idx]

NUM_CLIENTS, FL_ROUNDS, LOCAL_EPOCHS = 5, 10, 3

def train_local_classical(client_X, client_y, global_params, in_dim,
                          epochs=3, lr=1e-3, batch=64, proximal_mu=0.0, seed=42):
    set_global_seed(seed)
    m = FraudDetectorNet(in_dim).to(device)
    for p, ar in zip(m.parameters(), global_params):
        p.data.copy_(torch.tensor(ar, dtype=torch.float32).to(device))
    global_tensors = [p.detach().clone() for p in m.parameters()] if proximal_mu > 0 else None
    crit = nn.BCELoss(); opt = optim.AdamW(m.parameters(), lr=lr)
    _drop = (len(client_X) % batch == 1)   # BatchNorm1d needs >1 sample per batch in train mode
    ld = DataLoader(TensorDataset(torch.tensor(client_X, dtype=torch.float32),
                                  torch.tensor(client_y, dtype=torch.float32)),
                    batch_size=batch, shuffle=True, drop_last=_drop)
    m.train()
    for _ in range(epochs):
        for Xb, yb in ld:
            Xb, yb = Xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = crit(m(Xb), yb)
            if proximal_mu > 0:
                prox = sum(((p - g.detach())**2).sum()
                           for p, g in zip(m.parameters(), global_tensors))
                loss = loss + (proximal_mu/2.0) * prox
            loss.backward()
            nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            opt.step()
    return [p.detach().cpu().numpy() for p in m.parameters()]

def run_fl_classical(d, name, seed, algorithm='FedAvg',
                     mode='iid', proximal_mu=0.0):
    set_global_seed(seed)
    in_dim = d['X_tr_c'].shape[1]
    g = FraudDetectorNet(in_dim).to(device)
    g_params = [p.detach().cpu().numpy() for p in g.parameters()]
    clients = split_clients(d['X_tr_c'], d['y_tr_c'], NUM_CLIENTS, mode=mode,
                            alpha=0.5, seed=seed)
    weights = [len(c[0]) for c in clients]
    t_start = time.time()
    for r in range(1, FL_ROUNDS+1):
        local_params = [
            train_local_classical(Xc, yc, g_params, in_dim,
                                  epochs=LOCAL_EPOCHS, lr=1e-3,
                                  proximal_mu=proximal_mu, seed=seed+r)
            for (Xc, yc) in clients]
        g_params = weighted_fedavg_arrays(local_params, weights)
        for p, ar in zip(g.parameters(), g_params):
            p.data.copy_(torch.tensor(ar, dtype=torch.float32).to(device))
    g.eval()
    with torch.no_grad():
        p = g(torch.tensor(d['X_te_c'], dtype=torch.float32).to(device)).cpu().numpy()
    pred = (p >= 0.5).astype(int)
    return {
        'Dataset': name, 'Family': 'Federated',
        'Model': f'{algorithm}-{mode.upper()}', 'Seed': seed,
        'ROC-AUC' : roc_auc_score(d['y_te_c'], p),
        'F1-Score': f1_score(d['y_te_c'], pred),
        'Avg-Prec': average_precision_score(d['y_te_c'], p),
        'Params'  : sum(p_.numel() for p_ in g.parameters()),
        'Time (s)': round(time.time()-t_start, 1),
    }


In [14]:
# ─── Proposed Hybrid QDFL: QDCN-FedAvg with quantum-parameter aggregation ─
N_ENT_LAYERS_QDFL = 3   # Deeper than standalone QDCN — matches VQC depth

def build_qdfl_client(seed):
    """Same architecture as QDCN but with 3 entangling layers per channel."""
    set_global_seed(seed)
    dev_a = qml.device('lightning.qubit', wires=N_Q_CH)
    dev_b = qml.device('lightning.qubit', wires=N_Q_CH)
    @qml.qnode(dev_a, interface='torch', diff_method='best')
    def channel_a(inputs, weights):
        qml.AngleEmbedding(inputs, wires=range(N_Q_CH), rotation='Y')
        qml.BasicEntanglerLayers(weights, wires=range(N_Q_CH))
        return [qml.expval(qml.PauliZ(i)) for i in range(N_Q_CH)]
    @qml.qnode(dev_b, interface='torch', diff_method='best')
    def channel_b(inputs, weights):
        qml.AngleEmbedding(inputs, wires=range(N_Q_CH), rotation='X')
        for i in range(N_Q_CH-1):
            qml.CNOT(wires=[i, i+1])
        qml.BasicEntanglerLayers(weights, wires=range(N_Q_CH))
        return [qml.expval(qml.PauliY(i)) for i in range(N_Q_CH)]
    sa = {'weights': qml.BasicEntanglerLayers.shape(n_layers=N_ENT_LAYERS_QDFL, n_wires=N_Q_CH)}
    sb = {'weights': qml.BasicEntanglerLayers.shape(n_layers=N_ENT_LAYERS_QDFL, n_wires=N_Q_CH)}
    qa = qml.qnn.TorchLayer(channel_a, sa)
    qb = qml.qnn.TorchLayer(channel_b, sb)
    class QDFLClient(nn.Module):
        def __init__(self):
            super().__init__()
            self.bn     = nn.BatchNorm1d(N_QUBITS)
            self.q_a    = qa
            self.q_b    = qb
            self.post_a = nn.Sequential(nn.Linear(N_Q_CH, 16), nn.GELU(), nn.Dropout(0.2))
            self.post_b = nn.Sequential(nn.Linear(N_Q_CH, 16), nn.GELU(), nn.Dropout(0.2))
            self.fusion = nn.Sequential(
                nn.Linear(32, 64),  nn.GELU(),  nn.Dropout(0.3),
                nn.Linear(64, 32),  nn.GELU(),  nn.Dropout(0.2),
                nn.Linear(32, 16),  nn.GELU(),  nn.Dropout(0.1),
                nn.Linear(16,  1),  nn.Sigmoid())
        def forward(self, x):
            x  = self.bn(x); xa = x[:, :N_Q_CH]; xb = x[:, N_Q_CH:]
            return self.fusion(torch.cat(
                [self.post_a(self.q_a(xa)), self.post_b(self.q_b(xb))], -1)
            ).squeeze(-1)
    return QDFLClient().to(device)

In [15]:
def train_local_qdfl(client_X, client_y, global_state, seed,
                     epochs=3, lr=5e-4, batch=32):
    """Local training of QDCN client; returns both quantum and classical
    parameter tensors (entire state dict)."""
    set_global_seed(seed)
    m = build_qdfl_client(seed)
    m.load_state_dict(global_state)
    crit = nn.BCELoss()
    opt  = optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4)
    sch  = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    _drop = (len(client_X) % batch == 1)   # BatchNorm1d needs >1 sample per batch in train mode
    ld = DataLoader(TensorDataset(torch.tensor(client_X, dtype=torch.float32),
                                  torch.tensor(client_y, dtype=torch.float32)),
                    batch_size=batch, shuffle=True, drop_last=_drop)
    m.train()
    for _ in range(epochs):
        for Xb, yb in ld:
            Xb, yb = Xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = crit(m(Xb), yb); loss.backward()
            nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            opt.step()
        sch.step()
    return {k: v.detach().cpu().clone() for k, v in m.state_dict().items()}

def weighted_fedavg_state(state_list, weights):
    """Sample-weighted aggregation of full state dicts (quantum + classical)."""
    w_sum = sum(weights)
    out = {}
    for k in state_list[0]:
        if state_list[0][k].dtype.is_floating_point:
            out[k] = sum(w * s[k] for w, s in zip(weights, state_list)) / w_sum
        else:  # integer buffers (e.g. BN num_batches_tracked) — take first
            out[k] = state_list[0][k]
    return out

def run_qdfl_hybrid(d, name, seed, mode='iid'):
    """Run the proposed hybrid QDFL training loop."""
    set_global_seed(seed)
    t0 = time.time()
    # Initialise global model on the angle-normalised classical pool
    g = build_qdfl_client(seed)
    global_state = {k: v.detach().cpu().clone() for k, v in g.state_dict().items()}
    # Split angle-normalised classical pool into clients
    clients = split_clients(d['X_tr_qc'], d['y_tr_qc'], NUM_CLIENTS, mode=mode,
                            alpha=0.5, seed=seed)
    weights = [len(c[0]) for c in clients]
    round_aucs = []
    for r in range(1, FL_ROUNDS+1):
        local_states = [
            train_local_qdfl(Xc, yc, global_state, seed+r,
                             epochs=LOCAL_EPOCHS, lr=5e-4, batch=32)
            for (Xc, yc) in clients]
        global_state = weighted_fedavg_state(local_states, weights)
        g.load_state_dict(global_state)
        g.eval()
        with torch.no_grad():
            p = g(torch.tensor(d['X_te_qc'], dtype=torch.float32).to(device)).cpu().numpy()
        rnd_auc = roc_auc_score(d['y_te_qc'], p)
        round_aucs.append(rnd_auc)
        if r in (1, 5, 10):
            print(f'  Round {r:>2}/{FL_ROUNDS} | global AUC = {rnd_auc:.4f}')
    # Final evaluation
    g.eval()
    with torch.no_grad():
        p = g(torch.tensor(d['X_te_qc'], dtype=torch.float32).to(device)).cpu().numpy()
    pred = (p >= 0.5).astype(int)
    return {
        'Dataset': name, 'Family': 'Quantum-Federated',
        'Model': f'QDFL-Hybrid-{mode.upper()}', 'Seed': seed,
        'ROC-AUC' : roc_auc_score(d['y_te_qc'], p),
        'F1-Score': f1_score(d['y_te_qc'], pred),
        'Avg-Prec': average_precision_score(d['y_te_qc'], p),
        'Params'  : sum(p_.numel() for p_ in g.parameters()),
        'Time (s)': round(time.time()-t0, 1),
        'round_aucs': round_aucs,
    }, g

## Part C — Cross-validation utilities

In [16]:
# ============ Cross-validation utilities (LEAKAGE-SAFE) ============
# CHANGED: the OLD cv_pools returned ALREADY-TRANSFORMED pools (scaler/PCA/MinMax fit on
# the full pool), then folds split those. This version returns RAW rows per family; the
# preprocessor is refit INSIDE each fold (fit_fold) and, for LOCO, on training clusters only.
import numpy as np, pandas as pd
from sklearn.model_selection import (StratifiedKFold, RepeatedStratifiedKFold,
    StratifiedShuffleSplit, GroupKFold, LeaveOneGroupOut)
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
try:
    from IPython.display import display
except Exception:
    display = print

_FAMILY = {'q': ('angle', 8), 'qs': ('angle', 4), 'c': ('std', None), 'qc': ('angle', 8)}
_POOL_N = {'q': SAMPLE_QUANT, 'qs': SAMPLE_QSVM, 'c': SAMPLE_CLASS, 'qc': SAMPLE_CLASS}

def _subsample_df(df_eng, n, seed):
    fr = df_eng[df_eng[TARGET]==1].sample(n=min(n//2, (df_eng[TARGET]==1).sum()), random_state=seed).index
    lg = df_eng[df_eng[TARGET]==0].sample(n=min(n//2, (df_eng[TARGET]==0).sum()), random_state=seed).index
    return df_eng.loc[fr.tolist()+lg.tolist()].sample(frac=1, random_state=seed).reset_index(drop=True)

def cv_raw(name, key):
    """Raw (untransformed) balanced rows for a model family + its (mode, n_components)."""
    mode, ncomp = _FAMILY[key]
    return _subsample_df(DATASETS_ENG[name], _POOL_N[key], SEED), mode, ncomp

def fit_fold(raw, tr, te, mode, ncomp, seed):
    """Fit LeakageSafePreprocessor on the TRAIN fold only; transform train + val folds."""
    pp = LeakageSafePreprocessor(mode, ncomp, seed).fit(raw.iloc[tr])
    return pp.transform(raw.iloc[tr]), pp.transform(raw.iloc[te])

def loco_clients(raw, tr, groups, mode, ncomp, seed):
    """LOCO: fit preprocessor on TRAIN clusters only; return per-client transformed shards + pp."""
    pp = LeakageSafePreprocessor(mode, ncomp, seed).fit(raw.iloc[tr])
    gtr = groups[tr]; y = raw[TARGET].values
    clients = [(pp.transform(raw.iloc[tr][gtr==c]), y[tr][gtr==c]) for c in np.unique(gtr)]
    return clients, pp

def cluster_groups(raw, n_clusters, seed):
    """KMeans client partition on RAW standardized features (no PCA leak into definition)."""
    Xr = StandardScaler().fit_transform(raw[ROWWISE].values.astype(np.float32))
    return KMeans(n_clusters=n_clusters, random_state=seed, n_init=10).fit_predict(Xr)

def _metrics(y_true, prob):
    y_true = np.asarray(y_true); prob = np.asarray(prob); pred = (prob >= 0.5).astype(int)
    if len(np.unique(y_true)) < 2:
        return (np.nan, f1_score(y_true, pred, zero_division=0), np.nan)
    return (roc_auc_score(y_true, prob), f1_score(y_true, pred), average_precision_score(y_true, prob))

def cv_summary(rows):
    df = pd.DataFrame(rows)
    g = (df.groupby(['Dataset','Family','Model','CV'])
           .agg(folds=('ROC-AUC','size'), ROC_AUC_mean=('ROC-AUC','mean'), ROC_AUC_std=('ROC-AUC','std'),
                F1_mean=('F1-Score','mean'), F1_std=('F1-Score','std'),
                AP_mean=('Avg-Prec','mean'), AP_std=('Avg-Prec','std')).reset_index())
    for c in ['ROC_AUC_mean','ROC_AUC_std','F1_mean','F1_std','AP_mean','AP_std']: g[c]=g[c].round(4)
    return g

CV_RESULTS = []
print('Leakage-safe CV utilities ready (preprocessing refit per fold).')


Leakage-safe CV utilities ready (preprocessing refit per fold).


## 1. Classical ML — Stratified K-Fold + Repeated Stratified K-Fold

In [17]:
# ============ 1) CLASSICAL ML : Stratified K-Fold + Repeated Stratified K-Fold (LEAKAGE-SAFE) ============
N_SPLITS_CLS, N_REPEATS_CLS, NN_EPOCHS_CLS = 5, 3, 25

def _make_classical_clfs(seed):
    return {
        'Logistic Regression': LogisticRegression(C=1.0, max_iter=2000, class_weight='balanced', random_state=seed, n_jobs=-1),
        'Random Forest'      : RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=5, class_weight='balanced', n_jobs=-1, random_state=seed),
        'XGBoost'            : xgb.XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.08, subsample=0.9, colsample_bytree=0.9, eval_metric='logloss', n_jobs=-1, random_state=seed),
    }
def _fit_nn_fold(Xtr, ytr, Xte, seed, epochs=NN_EPOCHS_CLS):
    set_global_seed(seed); model = SimpleClassicalNN(Xtr.shape[1]).to(device)
    crit, opt = nn.BCELoss(), optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    _drop = (len(Xtr) % 64 == 1)
    ld = DataLoader(TensorDataset(torch.tensor(Xtr, dtype=torch.float32), torch.tensor(ytr, dtype=torch.float32)), batch_size=64, shuffle=True, drop_last=_drop)
    model.train()
    for _ in range(epochs):
        for Xb, yb in ld:
            Xb, yb = Xb.to(device), yb.to(device); opt.zero_grad(); crit(model(Xb), yb).backward(); opt.step()
    model.eval()
    with torch.no_grad():
        return model(torch.tensor(Xte, dtype=torch.float32).to(device)).cpu().numpy()

def run_classical_cv():
    rows, schemes = [], [
        ('StratifiedKFold',         StratifiedKFold(n_splits=N_SPLITS_CLS, shuffle=True, random_state=SEED)),
        ('RepeatedStratifiedKFold', RepeatedStratifiedKFold(n_splits=N_SPLITS_CLS, n_repeats=N_REPEATS_CLS, random_state=SEED)),
    ]
    for name in DATA:
        raw, mode, ncomp = cv_raw(name, 'c'); y = raw[TARGET].values
        for cv_name, splitter in schemes:
            for fold, (tr, te) in enumerate(splitter.split(np.arange(len(raw)), y)):
                Xtr, Xte = fit_fold(raw, tr, te, mode, ncomp, SEED + fold)   # per-fold train-only fit
                for mname, clf in _make_classical_clfs(SEED + fold).items():
                    clf.fit(Xtr, y[tr])
                    a, f, ap = _metrics(y[te], clf.predict_proba(Xte)[:, 1])
                    rows.append(dict(Dataset=name, Family='Classical', Model=mname, CV=cv_name, fold=fold, **{'ROC-AUC': a, 'F1-Score': f, 'Avg-Prec': ap}))
                a, f, ap = _metrics(y[te], _fit_nn_fold(Xtr, y[tr], Xte, SEED + fold))
                rows.append(dict(Dataset=name, Family='Classical', Model='Neural Network', CV=cv_name, fold=fold, **{'ROC-AUC': a, 'F1-Score': f, 'Avg-Prec': ap}))
            print(f'  [{cv_name:24s}] {name:24s} done')
    return rows

print('=' * 70, '\nCLASSICAL ML CROSS-VALIDATION (leakage-safe)\n', '=' * 70, sep='')
classical_rows = run_classical_cv(); CV_RESULTS += classical_rows
display(cv_summary(classical_rows))


CLASSICAL ML CROSS-VALIDATION (leakage-safe)
  [StratifiedKFold         ] Primary (Azamuke 2024)   done
  [RepeatedStratifiedKFold ] Primary (Azamuke 2024)   done
  [StratifiedKFold         ] PaySim (Lopez-Rojas)     done
  [RepeatedStratifiedKFold ] PaySim (Lopez-Rojas)     done


,Dataset,Family,Model,CV,folds,ROC_AUC_mean,ROC_AUC_std,F1_mean,F1_std,AP_mean,AP_std
0,PaySim (Lopez-Rojas),Classical,Logistic Regression,RepeatedStratifiedKFold,15,0.9902,0.0011,0.9497,0.0026,0.9912,0.0009
1,PaySim (Lopez-Rojas),Classical,Logistic Regression,StratifiedKFold,5,0.9901,0.0011,0.9499,0.0026,0.9911,0.0010
2,PaySim (Lopez-Rojas),Classical,Neural Network,RepeatedStratifiedKFold,15,0.9972,0.0005,0.9762,0.0027,0.9974,0.0004
3,PaySim (Lopez-Rojas),Classical,Neural Network,StratifiedKFold,5,0.9971,0.0005,0.9745,0.0024,0.9972,0.0005
4,PaySim (Lopez-Rojas),Classical,Random Forest,RepeatedStratifiedKFold,15,0.9993,0.0003,0.9910,0.0019,0.9994,0.0002
5,PaySim (Lopez-Rojas),Classical,Random Forest,StratifiedKFold,5,0.9993,0.0003,0.9909,0.0020,0.9994,0.0002
6,PaySim (Lopez-Rojas),Classical,XGBoost,RepeatedStratifiedKFold,15,0.9993,0.0003,0.9938,0.0016,0.9994,0.0002
7,PaySim (Lopez-Rojas),Classical,XGBoost,StratifiedKFold,5,0.9993,0.0003,0.9938,0.0021,0.9994,0.0002
8,Primary (Azamuke 2024),Classical,Logistic Regression,RepeatedStratifiedKFold,15,0.8683,0.0058,0.8845,0.0037,0.7741,0.0095
9,Primary (Azamuke 2024),Classical,Logistic Regression,StratifiedKFold,5,0.8683,0.0067,0.8843,0.0039,0.7742,0.0122


## 2. Quantum ML — Repeated K-Fold + Monte-Carlo Holdout

Simulator-bound, so sub-samples and fold counts are deliberately small (tune the constants up if you have GPU time).

In [18]:
# ============ 2) QUANTUM ML : Repeated K-Fold + Monte-Carlo Holdout (LEAKAGE-SAFE) ============
QCV_SAMPLE, QCV_EPOCHS        = 1000, 12
QCV_KF_SPLITS, QCV_KF_REPEATS = 3, 2
QCV_MC_SPLITS                 = 4
QSVM_CV_SAMPLE                = 300
QSVM_KF_SPLITS, QSVM_MC_SPLITS = 3, 3

def _train_quantum_fold(builder, Xtr, ytr, Xte, seed, epochs=QCV_EPOCHS, tag='Q'):
    set_global_seed(seed); m = builder(seed)
    Xt, Xv, yt, yv = train_test_split(Xtr, ytr, test_size=0.15, stratify=ytr, random_state=seed)
    tr_ld, va_ld = make_loaders(Xt, yt, Xv, yv, batch=32)
    train_quantum(m, tr_ld, va_ld, epochs=epochs, lr=5e-4, tag=tag, verbose=False); m.eval()
    with torch.no_grad():
        return m(torch.tensor(Xte, dtype=torch.float32).to(device)).cpu().numpy()
def _qsvm_fold(Xtr, ytr, Xte, seed):
    Kfn = build_qsvm_kernel(seed); clf = SVC(kernel='precomputed', C=1.0, probability=True, random_state=seed)
    clf.fit(Kfn(Xtr, Xtr), ytr); return clf.predict_proba(Kfn(Xte, Xtr))[:, 1]

def run_quantum_cv():
    rows = []
    for mname, builder in [('VQC', build_vqc), ('QDCN', build_qdcn)]:
        for name in DATA:
            pool, mode, ncomp = cv_raw(name, 'q'); raw = _subsample_df(pool, QCV_SAMPLE, SEED); y = raw[TARGET].values
            schemes = [('RepeatedKFold', RepeatedStratifiedKFold(n_splits=QCV_KF_SPLITS, n_repeats=QCV_KF_REPEATS, random_state=SEED)),
                       ('MonteCarloHoldout', StratifiedShuffleSplit(n_splits=QCV_MC_SPLITS, test_size=0.25, random_state=SEED))]
            for cv_name, splitter in schemes:
                for fold, (tr, te) in enumerate(splitter.split(np.arange(len(raw)), y)):
                    Xtr, Xte = fit_fold(raw, tr, te, mode, ncomp, SEED + fold)
                    a, f, ap = _metrics(y[te], _train_quantum_fold(builder, Xtr, y[tr], Xte, SEED + fold, tag=mname))
                    rows.append(dict(Dataset=name, Family='Quantum', Model=mname, CV=cv_name, fold=fold, **{'ROC-AUC': a, 'F1-Score': f, 'Avg-Prec': ap}))
                print(f'  [{mname:5s}|{cv_name:18s}] {name:24s} done')
    for name in DATA:
        pool, mode, ncomp = cv_raw(name, 'qs'); raw = _subsample_df(pool, QSVM_CV_SAMPLE, SEED); y = raw[TARGET].values
        schemes = [('RepeatedKFold', RepeatedStratifiedKFold(n_splits=QSVM_KF_SPLITS, n_repeats=1, random_state=SEED)),
                   ('MonteCarloHoldout', StratifiedShuffleSplit(n_splits=QSVM_MC_SPLITS, test_size=0.25, random_state=SEED))]
        for cv_name, splitter in schemes:
            for fold, (tr, te) in enumerate(splitter.split(np.arange(len(raw)), y)):
                Xtr, Xte = fit_fold(raw, tr, te, mode, ncomp, SEED + fold)
                a, f, ap = _metrics(y[te], _qsvm_fold(Xtr, y[tr], Xte, SEED + fold))
                rows.append(dict(Dataset=name, Family='Quantum', Model='QSVM', CV=cv_name, fold=fold, **{'ROC-AUC': a, 'F1-Score': f, 'Avg-Prec': ap}))
            print(f'  [QSVM |{cv_name:18s}] {name:24s} done')
    return rows

print('=' * 70, '\nQUANTUM ML CROSS-VALIDATION (leakage-safe; simulator)\n', '=' * 70, sep='')
quantum_rows = run_quantum_cv(); CV_RESULTS += quantum_rows
display(cv_summary(quantum_rows))


QUANTUM ML CROSS-VALIDATION (leakage-safe; simulator)
  [VQC  |RepeatedKFold     ] Primary (Azamuke 2024)   done
  [VQC  |MonteCarloHoldout ] Primary (Azamuke 2024)   done
  [VQC  |RepeatedKFold     ] PaySim (Lopez-Rojas)     done
  [VQC  |MonteCarloHoldout ] PaySim (Lopez-Rojas)     done
  [QDCN |RepeatedKFold     ] Primary (Azamuke 2024)   done
  [QDCN |MonteCarloHoldout ] Primary (Azamuke 2024)   done
  [QDCN |RepeatedKFold     ] PaySim (Lopez-Rojas)     done
  [QDCN |MonteCarloHoldout ] PaySim (Lopez-Rojas)     done
  [QSVM |RepeatedKFold     ] Primary (Azamuke 2024)   done
  [QSVM |MonteCarloHoldout ] Primary (Azamuke 2024)   done
  [QSVM |RepeatedKFold     ] PaySim (Lopez-Rojas)     done
  [QSVM |MonteCarloHoldout ] PaySim (Lopez-Rojas)     done


,Dataset,Family,Model,CV,folds,ROC_AUC_mean,ROC_AUC_std,F1_mean,F1_std,AP_mean,AP_std
0,PaySim (Lopez-Rojas),Quantum,QDCN,MonteCarloHoldout,4,0.8192,0.0652,0.6884,0.1196,0.8206,0.0668
1,PaySim (Lopez-Rojas),Quantum,QDCN,RepeatedKFold,6,0.7616,0.1092,0.6416,0.2020,0.7646,0.1209
2,PaySim (Lopez-Rojas),Quantum,QSVM,MonteCarloHoldout,3,0.9440,0.0134,0.8669,0.0288,0.9416,0.0149
3,PaySim (Lopez-Rojas),Quantum,QSVM,RepeatedKFold,3,0.9476,0.0153,0.8585,0.0096,0.9507,0.0150
4,PaySim (Lopez-Rojas),Quantum,VQC,MonteCarloHoldout,4,0.5256,0.1605,0.5000,0.3333,0.5396,0.1254
5,PaySim (Lopez-Rojas),Quantum,VQC,RepeatedKFold,6,0.5407,0.1716,0.4426,0.3429,0.5540,0.1355
6,Primary (Azamuke 2024),Quantum,QDCN,MonteCarloHoldout,4,0.7598,0.0640,0.6641,0.1263,0.7349,0.0507
7,Primary (Azamuke 2024),Quantum,QDCN,RepeatedKFold,6,0.7239,0.0610,0.5367,0.3095,0.6988,0.0594
8,Primary (Azamuke 2024),Quantum,QSVM,MonteCarloHoldout,3,0.8845,0.0129,0.8518,0.0227,0.8317,0.0307
9,Primary (Azamuke 2024),Quantum,QSVM,RepeatedKFold,3,0.8687,0.0445,0.8515,0.0485,0.8178,0.0535


## 3. Federated Learning — Group K-Fold + LOCO-CV

Clusters are discovered with K-Means. **LOCO-CV** (leave-one-cluster-out) is the key test: the federated model trains on the remaining clusters (each cluster acts as one client) and is evaluated on the unseen cluster.

In [19]:
# ============ 3) FEDERATED LEARNING : Group K-Fold + LOCO-CV (LEAKAGE-SAFE) ============
FLCV_SAMPLE, N_CLUSTERS_FL, FL_ROUNDS_CV = 6000, 5, 6

def _fedavg_train_eval(clients, in_dim, Xte, seed, rounds=FL_ROUNDS_CV):
    set_global_seed(seed); g = FraudDetectorNet(in_dim).to(device)
    g_params = [p.detach().cpu().numpy() for p in g.parameters()]; weights = [len(cx) for cx, _ in clients]
    for r in range(1, rounds + 1):
        local = [train_local_classical(cx, cy, g_params, in_dim, epochs=LOCAL_EPOCHS, lr=1e-3, seed=seed + r) for cx, cy in clients]
        g_params = weighted_fedavg_arrays(local, weights)
    for p, ar in zip(g.parameters(), g_params): p.data.copy_(torch.tensor(ar, dtype=torch.float32).to(device))
    g.eval()
    with torch.no_grad():
        return g(torch.tensor(Xte, dtype=torch.float32).to(device)).cpu().numpy()

def run_federated_cv():
    rows = []
    for name in DATA:
        pool, mode, ncomp = cv_raw(name, 'c'); raw = _subsample_df(pool, FLCV_SAMPLE, SEED); y = raw[TARGET].values
        groups = cluster_groups(raw, N_CLUSTERS_FL, SEED); in_dim = len(FEATURE_ORDER)
        # (a) Group K-Fold : training shard re-split into NUM_CLIENTS IID clients
        for fold, (tr, te) in enumerate(GroupKFold(n_splits=N_CLUSTERS_FL).split(np.arange(len(raw)), y, groups)):
            Xtr, Xte = fit_fold(raw, tr, te, mode, ncomp, SEED + fold)       # per-fold train-only fit
            clients = split_clients(Xtr, y[tr], NUM_CLIENTS, mode='iid', seed=SEED + fold)
            a, f, ap = _metrics(y[te], _fedavg_train_eval(clients, in_dim, Xte, SEED + fold))
            rows.append(dict(Dataset=name, Family='Federated', Model='FedAvg', CV='GroupKFold', fold=fold, **{'ROC-AUC': a, 'F1-Score': f, 'Avg-Prec': ap}))
        print(f'  [GroupKFold] {name:24s} done')
        # (b) LOCO-CV : preprocessing fit on TRAIN clusters only; held-out cluster transformed
        for fold, (tr, te) in enumerate(LeaveOneGroupOut().split(np.arange(len(raw)), y, groups)):
            clients, pp = loco_clients(raw, tr, groups, mode, ncomp, SEED + fold)
            Xte = pp.transform(raw.iloc[te])
            a, f, ap = _metrics(y[te], _fedavg_train_eval(clients, in_dim, Xte, SEED + fold))
            rows.append(dict(Dataset=name, Family='Federated', Model='FedAvg', CV='LOCO-CV', fold=fold, held_out_cluster=int(groups[te][0]), **{'ROC-AUC': a, 'F1-Score': f, 'Avg-Prec': ap}))
        print(f'  [LOCO-CV   ] {name:24s} done (each cluster held out once)')
    return rows

print('=' * 70, '\nFEDERATED LEARNING CROSS-VALIDATION (leakage-safe)\n', '=' * 70, sep='')
federated_rows = run_federated_cv(); CV_RESULTS += federated_rows
display(cv_summary(federated_rows))


FEDERATED LEARNING CROSS-VALIDATION (leakage-safe)
  [GroupKFold] Primary (Azamuke 2024)   done
  [LOCO-CV   ] Primary (Azamuke 2024)   done (each cluster held out once)
  [GroupKFold] PaySim (Lopez-Rojas)     done
  [LOCO-CV   ] PaySim (Lopez-Rojas)     done (each cluster held out once)


,Dataset,Family,Model,CV,folds,ROC_AUC_mean,ROC_AUC_std,F1_mean,F1_std,AP_mean,AP_std
0,PaySim (Lopez-Rojas),Federated,FedAvg,GroupKFold,5,0.8736,0.1447,0.5168,0.4717,0.6616,0.3994
1,PaySim (Lopez-Rojas),Federated,FedAvg,LOCO-CV,5,0.6664,0.3501,0.2172,0.4379,0.4233,0.4992
2,Primary (Azamuke 2024),Federated,FedAvg,GroupKFold,5,0.8459,0.0080,0.3446,0.4721,0.8283,0.0105
3,Primary (Azamuke 2024),Federated,FedAvg,LOCO-CV,5,0.8552,0.0212,0.3526,0.4830,0.8344,0.0213


## 4. QDFL (proposed) — LOCO-CV + Stratified-Federated-CV combo

The quantum-federated model aggregates the **quantum-circuit parameters** across clients. It is evaluated three ways: **LOCO-CV** (remaining clusters are the quantum clients), **Stratified-Federated-CV** (class-balanced clients inside a stratified outer fold), and the **combo** (leave one cluster out for the test, split the remainder into stratified clients). This is the heaviest section.

In [20]:
# ============ 4) QDFL : LOCO-CV + Stratified-Federated-CV combo (LEAKAGE-SAFE) ============
QDFLCV_SAMPLE, N_CLUSTERS_QDFL, QDFL_ROUNDS_CV = 1200, 3, 4

def _qdfl_train_eval(clients, Xte, seed, rounds=QDFL_ROUNDS_CV):
    set_global_seed(seed); g = build_qdfl_client(seed)
    global_state = {k: v.detach().cpu().clone() for k, v in g.state_dict().items()}; weights = [len(cx) for cx, _ in clients]
    for r in range(1, rounds + 1):
        local_states = [train_local_qdfl(cx, cy, global_state, seed + r, epochs=LOCAL_EPOCHS, lr=5e-4, batch=32) for cx, cy in clients]
        global_state = weighted_fedavg_state(local_states, weights)
    g.load_state_dict(global_state); g.eval()
    with torch.no_grad():
        return g(torch.tensor(Xte, dtype=torch.float32).to(device)).cpu().numpy()
def _stratified_clients_arr(X, y, n_clients, seed):
    skf = StratifiedKFold(n_splits=n_clients, shuffle=True, random_state=seed)
    return [(X[idx], y[idx]) for _, idx in skf.split(X, y)]

def run_qdfl_cv():
    rows = []
    for name in DATA:
        pool, mode, ncomp = cv_raw(name, 'qc'); raw = _subsample_df(pool, QDFLCV_SAMPLE, SEED); y = raw[TARGET].values
        groups = cluster_groups(raw, N_CLUSTERS_QDFL, SEED)
        # (a) LOCO-CV : preprocessing fit on TRAIN clusters only
        for fold, (tr, te) in enumerate(LeaveOneGroupOut().split(np.arange(len(raw)), y, groups)):
            clients, pp = loco_clients(raw, tr, groups, mode, ncomp, SEED + fold)
            Xte = pp.transform(raw.iloc[te])
            a, f, ap = _metrics(y[te], _qdfl_train_eval(clients, Xte, SEED + fold))
            rows.append(dict(Dataset=name, Family='Quantum-Federated', Model='QDFL', CV='LOCO-CV', fold=fold, held_out_cluster=int(groups[te][0]), **{'ROC-AUC': a, 'F1-Score': f, 'Avg-Prec': ap}))
        print(f'  [QDFL|LOCO-CV         ] {name:24s} done')
        # (b) Stratified-Federated-CV : preprocessing fit on TRAIN fold; stratified clients inside
        for fold, (tr, te) in enumerate(StratifiedKFold(n_splits=N_CLUSTERS_QDFL, shuffle=True, random_state=SEED).split(np.arange(len(raw)), y)):
            Xtr, Xte = fit_fold(raw, tr, te, mode, ncomp, SEED + fold)
            clients = _stratified_clients_arr(Xtr, y[tr], NUM_CLIENTS, SEED + fold)
            a, f, ap = _metrics(y[te], _qdfl_train_eval(clients, Xte, SEED + fold))
            rows.append(dict(Dataset=name, Family='Quantum-Federated', Model='QDFL', CV='StratifiedFedCV', fold=fold, **{'ROC-AUC': a, 'F1-Score': f, 'Avg-Prec': ap}))
        print(f'  [QDFL|StratifiedFedCV ] {name:24s} done')
        # (c) Combo : leave one cluster out (test) + STRATIFIED clients on remaining TRAIN data
        for fold, (tr, te) in enumerate(LeaveOneGroupOut().split(np.arange(len(raw)), y, groups)):
            pp = LeakageSafePreprocessor(mode, ncomp, SEED + fold).fit(raw.iloc[tr])
            Xtr, Xte = pp.transform(raw.iloc[tr]), pp.transform(raw.iloc[te])
            clients = _stratified_clients_arr(Xtr, y[tr], NUM_CLIENTS, SEED + fold)
            a, f, ap = _metrics(y[te], _qdfl_train_eval(clients, Xte, SEED + fold))
            rows.append(dict(Dataset=name, Family='Quantum-Federated', Model='QDFL', CV='LOCO+StratFed', fold=fold, held_out_cluster=int(groups[te][0]), **{'ROC-AUC': a, 'F1-Score': f, 'Avg-Prec': ap}))
        print(f'  [QDFL|LOCO+StratFed   ] {name:24s} done (combo)')
    return rows

print('=' * 70, '\nQDFL CROSS-VALIDATION (leakage-safe; slowest section)\n', '=' * 70, sep='')
qdfl_rows = run_qdfl_cv(); CV_RESULTS += qdfl_rows
display(cv_summary(qdfl_rows))


QDFL CROSS-VALIDATION (leakage-safe; slowest section)
  [QDFL|LOCO-CV         ] Primary (Azamuke 2024)   done
  [QDFL|StratifiedFedCV ] Primary (Azamuke 2024)   done
  [QDFL|LOCO+StratFed   ] Primary (Azamuke 2024)   done (combo)
  [QDFL|LOCO-CV         ] PaySim (Lopez-Rojas)     done
  [QDFL|StratifiedFedCV ] PaySim (Lopez-Rojas)     done
  [QDFL|LOCO+StratFed   ] PaySim (Lopez-Rojas)     done (combo)


,Dataset,Family,Model,CV,folds,ROC_AUC_mean,ROC_AUC_std,F1_mean,F1_std,AP_mean,AP_std
0,PaySim (Lopez-Rojas),Quantum-Federated,QDFL,LOCO+StratFed,3,0.5335,0.0813,0.0043,0.0074,0.5984,0.5118
1,PaySim (Lopez-Rojas),Quantum-Federated,QDFL,LOCO-CV,3,0.5376,0.2339,0.3288,0.5584,0.6230,0.5312
2,PaySim (Lopez-Rojas),Quantum-Federated,QDFL,StratifiedFedCV,3,0.7931,0.0928,0.0000,0.0000,0.7841,0.1010
3,Primary (Azamuke 2024),Quantum-Federated,QDFL,LOCO+StratFed,3,0.6459,0.3629,0.1944,0.3368,0.6236,0.2294
4,Primary (Azamuke 2024),Quantum-Federated,QDFL,LOCO-CV,3,0.7251,0.2633,0.2462,0.4265,0.6649,0.1775
5,Primary (Azamuke 2024),Quantum-Federated,QDFL,StratifiedFedCV,3,0.7948,0.0239,0.0000,0.0000,0.7528,0.0495


## 5. Combined cross-validation summary

In [21]:

# ============ Combined cross-validation summary (mean +/- std across folds) ============
full = cv_summary(CV_RESULTS)
pd.set_option('display.max_rows', None); pd.set_option('display.width', 160)
print('Cross-validated performance across all families:\n')
display(full)

disp = full.copy()
disp['ROC-AUC']  = disp.apply(lambda r: f"{r.ROC_AUC_mean:.4f} +/- {r.ROC_AUC_std:.4f}", axis=1)
disp['F1-Score'] = disp.apply(lambda r: f"{r.F1_mean:.4f} +/- {r.F1_std:.4f}", axis=1)
disp['Avg-Prec'] = disp.apply(lambda r: f"{r.AP_mean:.4f} +/- {r.AP_std:.4f}", axis=1)
display(disp[['Dataset', 'Family', 'Model', 'CV', 'folds', 'ROC-AUC', 'F1-Score', 'Avg-Prec']])

pd.DataFrame(CV_RESULTS).to_csv('qdfl_cv_per_fold.csv', index=False)
full.to_csv('qdfl_cv_summary.csv', index=False)
print('\nSaved per-fold scores -> qdfl_cv_per_fold.csv   and   summary -> qdfl_cv_summary.csv')


Cross-validated performance across all families:



,Dataset,Family,Model,CV,folds,ROC_AUC_mean,ROC_AUC_std,F1_mean,F1_std,AP_mean,AP_std
0,PaySim (Lopez-Rojas),Classical,Logistic Regression,RepeatedStratifiedKFold,15,0.9902,0.0011,0.9497,0.0026,0.9912,0.0009
1,PaySim (Lopez-Rojas),Classical,Logistic Regression,StratifiedKFold,5,0.9901,0.0011,0.9499,0.0026,0.9911,0.0010
2,PaySim (Lopez-Rojas),Classical,Neural Network,RepeatedStratifiedKFold,15,0.9972,0.0005,0.9762,0.0027,0.9974,0.0004
3,PaySim (Lopez-Rojas),Classical,Neural Network,StratifiedKFold,5,0.9971,0.0005,0.9745,0.0024,0.9972,0.0005
4,PaySim (Lopez-Rojas),Classical,Random Forest,RepeatedStratifiedKFold,15,0.9993,0.0003,0.9910,0.0019,0.9994,0.0002
5,PaySim (Lopez-Rojas),Classical,Random Forest,StratifiedKFold,5,0.9993,0.0003,0.9909,0.0020,0.9994,0.0002
6,PaySim (Lopez-Rojas),Classical,XGBoost,RepeatedStratifiedKFold,15,0.9993,0.0003,0.9938,0.0016,0.9994,0.0002
7,PaySim (Lopez-Rojas),Classical,XGBoost,StratifiedKFold,5,0.9993,0.0003,0.9938,0.0021,0.9994,0.0002
8,PaySim (Lopez-Rojas),Federated,FedAvg,GroupKFold,5,0.8736,0.1447,0.5168,0.4717,0.6616,0.3994
9,PaySim (Lopez-Rojas),Federated,FedAvg,LOCO-CV,5,0.6664,0.3501,0.2172,0.4379,0.4233,0.4992


,Dataset,Family,Model,CV,folds,ROC-AUC,F1-Score,Avg-Prec
0,PaySim (Lopez-Rojas),Classical,Logistic Regression,RepeatedStratifiedKFold,15,0.9902 +/- 0.0011,0.9497 +/- 0.0026,0.9912 +/- 0.0009
1,PaySim (Lopez-Rojas),Classical,Logistic Regression,StratifiedKFold,5,0.9901 +/- 0.0011,0.9499 +/- 0.0026,0.9911 +/- 0.0010
2,PaySim (Lopez-Rojas),Classical,Neural Network,RepeatedStratifiedKFold,15,0.9972 +/- 0.0005,0.9762 +/- 0.0027,0.9974 +/- 0.0004
3,PaySim (Lopez-Rojas),Classical,Neural Network,StratifiedKFold,5,0.9971 +/- 0.0005,0.9745 +/- 0.0024,0.9972 +/- 0.0005
4,PaySim (Lopez-Rojas),Classical,Random Forest,RepeatedStratifiedKFold,15,0.9993 +/- 0.0003,0.9910 +/- 0.0019,0.9994 +/- 0.0002
5,PaySim (Lopez-Rojas),Classical,Random Forest,StratifiedKFold,5,0.9993 +/- 0.0003,0.9909 +/- 0.0020,0.9994 +/- 0.0002
6,PaySim (Lopez-Rojas),Classical,XGBoost,RepeatedStratifiedKFold,15,0.9993 +/- 0.0003,0.9938 +/- 0.0016,0.9994 +/- 0.0002
7,PaySim (Lopez-Rojas),Classical,XGBoost,StratifiedKFold,5,0.9993 +/- 0.0003,0.9938 +/- 0.0021,0.9994 +/- 0.0002
8,PaySim (Lopez-Rojas),Federated,FedAvg,GroupKFold,5,0.8736 +/- 0.1447,0.5168 +/- 0.4717,0.6616 +/- 0.3994
9,PaySim (Lopez-Rojas),Federated,FedAvg,LOCO-CV,5,0.6664 +/- 0.3501,0.2172 +/- 0.4379,0.4233 +/- 0.4992



Saved per-fold scores -> qdfl_cv_per_fold.csv   and   summary -> qdfl_cv_summary.csv
